In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/dimlngkiu/cold-scenario/cold_items_test.json
/kaggle/input/datasets/dimlngkiu/cold-scenario/cold_items_val.json
/kaggle/input/datasets/dimlngkiu/cold-scenario/test_cold.parquet
/kaggle/input/datasets/dimlngkiu/cold-scenario/train_cold.parquet
/kaggle/input/datasets/dimlngkiu/cold-scenario/val_cold.parquet
/kaggle/input/datasets/dimlngkiu/lightgcn-backbone-cold/g_i.npy
/kaggle/input/datasets/dimlngkiu/lightgcn-backbone-cold/g_u.npy
/kaggle/input/datasets/dimlngkiu/lightgcn-backbone-cold/item_index.json
/kaggle/input/datasets/dimlngkiu/lightgcn-backbone-cold/user_index_cold.json
/kaggle/input/datasets/dimlngkiu/scores-aldi/user_ndcg20_aldi_fused.npy
/kaggle/input/datasets/dimlngkiu/scores-aldi/user_ndcg20_aldi_fused_3d.npy
/kaggle/input/datasets/dimlngkiu/scores-aldi/user_ndcg20_aldi_fused_3d_moi.npy
/kaggle/input/datasets/dimlngkiu/scores-aldi/user_ndcg20_aldi_fused_3d_trung.npy
/kaggle/input/datasets/dimlngkiu/scores-aldi/user_ndcg20_aldi_fused_moi.npy
/kaggle/in

In [2]:
import os, shutil

# Tạo cấu trúc thư mục src/ 
os.makedirs("src/losses", exist_ok=True)
os.makedirs("src/models", exist_ok=True)
os.makedirs("src/eval", exist_ok=True)

# Tạo các file __init__.py rỗng để Python nhận diện package
for pkg in ["src", "src/losses", "src/models", "src/eval"]:
    open(f"{pkg}/__init__.py", "a").close()

# Copy evaluator.py từ Kaggle Dataset vào src/eval/
INPUT_EVAL_PY = "/kaggle/input/datasets/dimlngkiu/evaluator/evaluator.py"

assert os.path.exists(INPUT_EVAL_PY), f"Không tìm thấy evaluator.py tại {INPUT_EVAL_PY}"
shutil.copy(INPUT_EVAL_PY, "src/eval/evaluator.py")

print("Đã khởi tạo xong cấu trúc thư mục!")

Đã khởi tạo xong cấu trúc thư mục!


In [3]:
%%writefile src/losses/align_loss.py

import torch
import torch.nn.functional as F

def align_loss(
    g: torch.Tensor,
    s: torch.Tensor,
    dup_group_id: torch.Tensor,
    tau: float = 0.2,
    use_false_neg_mask: bool = True,
) -> torch.Tensor:
    """
    Hàm mất mát căn chỉnh tương phản InfoNCE có che cặp âm giả theo dup_group_id.
    
    Tham số:
        g: [B_I, d] - Vector cộng tác g_i (CHƯA bị ID-dropout) của các khóa ấm DUY NHẤT trong batch.
        s: [B_I, d] - Vector ngữ nghĩa sau chiếu f(s_i) của các khóa ấm DUY NHẤT trong batch.
        dup_group_id: [B_I] - Mã nhóm trùng lặp (gộp bắc cầu trùng tên chuẩn hóa hoặc trùng mô tả).
        tau: Nhiệt độ InfoNCE (mặc định 0.2).
        use_false_neg_mask: Đặt True cho MVP chính; đặt False khi chạy Ablation "bỏ mặt nạ âm giả".
    """
    g_norm = F.normalize(g, dim=-1)
    s_norm = F.normalize(s, dim=-1)
    logits = (g_norm @ s_norm.t()) / tau

    if use_false_neg_mask:
        same_group = dup_group_id[:, None] == dup_group_id[None, :]
        eye = torch.eye(len(g), dtype=torch.bool, device=g.device)
        # Che các cặp khác mã khóa học (ngoài đường chéo) nhưng cùng dup_group_id
        logits = logits.masked_fill(same_group & ~eye, float("-inf"))

    targets = torch.arange(len(g), device=g.device)
    return F.cross_entropy(logits, targets)

Writing src/losses/align_loss.py


In [4]:
%%writefile src/models/mvp.py

import torch
import torch.nn as nn
import torch.nn.functional as F

class MVP(nn.Module):
    """
    Mô hình MVP kết hợp biểu diễn đồ thị LightGCN nền chung với nhánh chiếu ngữ nghĩa MLP 2 lớp,
    cơ chế ID-Dropout và căn chỉnh tương phản InfoNCE.
    """
    def __init__(
        self,
        g_u_init: torch.Tensor,
        g_i_init: torch.Tensor,
        sem: torch.Tensor,
        d: int = 64,
        p_drop: float = 0.3,
        freeze_backbone: bool = True,
    ):
        super().__init__()
        self.nu, self.ni = g_u_init.size(0), g_i_init.size(0)
        self.p = p_drop
        self.freeze_backbone = freeze_backbone

        # Đóng băng hoặc tinh chỉnh tiếp g_u, g_i từ LightGCN nền chung
        if freeze_backbone:
            self.register_buffer("g_u", g_u_init.clone().detach())
            self.register_buffer("g_i", g_i_init.clone().detach())
        else:
            self.g_u = nn.Parameter(g_u_init.clone().detach())
            self.g_i = nn.Parameter(g_i_init.clone().detach())

        # Vector văn bản đóng băng: [3769, 384] hoặc [3769, 768]
        self.register_buffer("sem", sem.clone().detach())

        # Mạng chiếu phi tuyến 2 lớp với GELU: d_t -> d -> d
        d_text = sem.size(1)
        self.proj = nn.Sequential(
            nn.Linear(d_text, d),
            nn.GELU(),
            nn.Linear(d, d),
        )

    def item_rep(self, cold_mask: torch.Tensor = None):
        """
        Trả về:
            h_item: [n_items, d] - Biểu diễn khóa học (dùng cho BPR loss khi train hoặc chấm điểm khi eval)
            s_proj: [n_items, d] - Biểu diễn ngữ nghĩa f(s_i) (dùng cho InfoNCE align_loss)
            g_raw:  [n_items, d] - Biểu diễn cộng tác g_i gốc chưa bị ID-dropout (dùng cho InfoNCE align_loss)
        """
        s_proj = self.proj(self.sem)
        g_raw = self.g_i
        g_used = g_raw

        if self.training and self.p > 0.0:
            keep = (torch.rand(g_used.size(0), 1, device=g_used.device) > self.p).float()
            g_used = g_used * keep

        if cold_mask is not None:
            g_used = g_used.masked_fill(cold_mask[:, None], 0.0)

        h_item = g_used + s_proj
        return h_item, s_proj, g_raw


def sample_neg(n: int, warm_items: torch.Tensor) -> torch.Tensor:
    """Lấy mẫu âm nghiêm ngặt trong tập khóa học ấm (I_w)."""
    idx = torch.randint(len(warm_items), (n,), device=warm_items.device)
    return warm_items[idx]


def bpr_loss(u_emb: torch.Tensor, h_pos: torch.Tensor, h_neg: torch.Tensor) -> torch.Tensor:
    """Hàm mất mát xếp hạng BPR."""
    pos_scores = (u_emb * h_pos).sum(dim=-1)
    neg_scores = (u_emb * h_neg).sum(dim=-1)
    return -F.logsigmoid(pos_scores - neg_scores).mean()

Writing src/models/mvp.py


In [5]:
import copy
import glob
import hashlib
import itertools
import json
import os
import random
import time
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
import torch
from src.eval.evaluator import evaluate, topk_random_ties
from src.losses.align_loss import align_loss
from src.models.mvp import MVP, bpr_loss, sample_neg

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seed(s: int):
  random.seed(s)
  np.random.seed(s)
  torch.manual_seed(s)
  torch.cuda.manual_seed_all(s)


def to_bool_mask(ids, c2i, n_items: int) -> torch.Tensor:
  m = torch.zeros(n_items, dtype=torch.bool)
  m[[c2i[str(c)] for c in ids]] = True
  return m


md5 = lambda p: hashlib.md5(open(p, "rb").read()).hexdigest()

ITEM_IDX_PATH = "/kaggle/input/datasets/dimlngkiu/lightgcn-backbone-cold/item_index.json"
USER_IDX_PATH = "/kaggle/input/datasets/dimlngkiu/lightgcn-backbone-cold/user_index_cold.json"
GU_PATH = "/kaggle/input/datasets/dimlngkiu/lightgcn-backbone-cold/g_u.npy"
GI_PATH = "/kaggle/input/datasets/dimlngkiu/lightgcn-backbone-cold/g_i.npy"

RAW_IDX_PATH = "/kaggle/input/datasets/dimlngkiu/course-llm-mvp/raw_embeddings_course_index.json"
LLM_IDX_PATH = "/kaggle/input/datasets/dimlngkiu/course-llm-mvp/embeddings_course_index.json"
RAW_NPY = "/kaggle/input/datasets/dimlngkiu/course-llm-mvp/raw_course_text_embeddings.npy"
LLM_NPY = "/kaggle/input/datasets/dimlngkiu/course-llm-mvp/llm_profile_embeddings.npy"
LLM_3D_NPY = "/kaggle/input/datasets/dimlngkiu/course-llm-mvp/llm_profile_no_prereq_followup_embeddings.npy"

COLD_VAL_JSON = "/kaggle/input/datasets/dimlngkiu/cold-scenario/cold_items_val.json"
COLD_TEST_JSON = "/kaggle/input/datasets/dimlngkiu/cold-scenario/cold_items_test.json"
TRAIN_PARQUET = "/kaggle/input/datasets/dimlngkiu/cold-scenario/train_cold.parquet"
VAL_PARQUET = "/kaggle/input/datasets/dimlngkiu/cold-scenario/val_cold.parquet"
TEST_PARQUET = "/kaggle/input/datasets/dimlngkiu/cold-scenario/test_cold.parquet"

# Assert đồng bộ bảng chỉ số
course_ids = json.load(open(ITEM_IDX_PATH))
users = json.load(open(USER_IDX_PATH))
raw_meta = json.load(open(RAW_IDX_PATH))
llm_meta = json.load(open(LLM_IDX_PATH))

assert len(course_ids) == 3769 and course_ids == sorted(course_ids), "item_index.json chưa chuẩn!"
assert course_ids == raw_meta["course_id"] == llm_meta["course_id"], "Lệch thứ tự course_id giữa các bảng!"

item2idx = {str(c): i for i, c in enumerate(course_ids)}
user2idx = {str(u): i for i, u in enumerate(users)}

cold_val_ids = json.load(open(COLD_VAL_JSON))
cold_test_ids = json.load(open(COLD_TEST_JSON))
assert len(cold_val_ids) == 251 and len(cold_test_ids) == 502, "Số lượng khóa lạnh val/test không đúng 251/502!"

cold_val_idx = sorted(item2idx[str(c)] for c in cold_val_ids)
cold_test_idx = sorted(item2idx[str(c)] for c in cold_test_ids)
cold_idx = sorted(cold_val_idx + cold_test_idx)

# Nạp dữ liệu tương tác và kiểm tra 6 assert bắt buộc
tr_cold = pd.read_parquet(TRAIN_PARQUET)
va_cold = pd.read_parquet(VAL_PARQUET)
te_cold = pd.read_parquet(TEST_PARQUET)

tr_items = set(tr_cold.course_id.astype(str))
cv, ct = set(map(str, cold_val_ids)), set(map(str, cold_test_ids))
assert not (cv & tr_items) and not (ct & tr_items) and not (cv & ct), "LỖI: Khóa lạnh không tách biệt!"
assert set(va_cold.course_id.astype(str)) <= cv, "LỖI: va_cold chứa khóa ngoài cold_val!"
assert set(te_cold.course_id.astype(str)) <= ct, "LỖI: te_cold chứa khóa ngoài cold_test!"
assert set(va_cold.user_id.astype(str)) <= set(tr_cold.user_id.astype(str)), "LỖI: va_cold có user lạ!"
assert set(te_cold.user_id.astype(str)) <= set(tr_cold.user_id.astype(str)), "LỖI: te_cold có user lạ!"

tr_df, va_df, te_df = tr_cold, va_cold, te_cold

u_tr = tr_df["user_id"].astype(str).map(user2idx).to_numpy()
i_tr = tr_df["course_id"].astype(str).map(item2idx).to_numpy()
assert not np.isnan(u_tr).any() and not np.isnan(i_tr).any(), "Có user/item trong train không nằm trong bảng index!"

# Tập khóa ấm thực tế xuất hiện trong train_cold (2.401 khóa có tương tác)
warm_items_np = np.unique(i_tr)
assert len(warm_items_np) == 2401 and len(set(warm_items_np).intersection(set(cold_idx))) == 0, "LỖI RÒ RỈ!"
warm_items_tensor = torch.tensor(warm_items_np, dtype=torch.long, device=DEVICE)

# Kiểm tra Backbone g_u, g_i bản mới và xác nhận 615 khóa ấm không tương tác = 0
g_u_np, g_i_np = np.load(GU_PATH).astype(np.float32), np.load(GI_PATH).astype(np.float32)
assert g_u_np.shape == (len(users), 64) and g_i_np.shape == (3769,64,), "Sai kích thước g_u hoặc g_i!"
assert np.allclose(g_i_np[cold_idx], 0.0), "LỖI RÒ RỈ: Khóa lạnh trong g_i khác 0!"

inactive_warm_idx = sorted(set(range(3769)) - set(warm_items_np.tolist()) - set(cold_idx))
assert len(inactive_warm_idx) == 615, f"Số khóa ấm không tương tác phải là 615, thực tế: {len(inactive_warm_idx)}"
assert np.allclose(g_i_np[inactive_warm_idx], 0.0), "LỖI: 615 khóa ấm không tương tác trong g_i chưa bằng 0!"

n_zero_gi = int((np.linalg.norm(g_i_np, axis=1) == 0).sum())
print("MD5 Checksums:")
for name, path in [
    ("g_u", GU_PATH),
    ("g_i", GI_PATH),
    ("raw_emb", RAW_NPY),
    ("llm_emb", LLM_NPY),
    ("llm_3d_emb", LLM_3D_NPY),
]:
  print(f"  {name:10s}: {md5(path)}")
print(
    f"  [Xác nhận] Bản g_i.npy trên đĩa đã có đúng {n_zero_gi}/3769 khóa có"
    " ||g_i|| = 0 (gồm 753 khóa lạnh + 615 khóa ấm không tương tác)!"
)

# Chuẩn bị vector đầu vào (384 chiều và 768 chiều)
S_raw = np.load(RAW_NPY).astype(np.float32)
S_llm = np.load(LLM_NPY).astype(np.float32)
S_llm_3d = np.load(LLM_3D_NPY).astype(np.float32)

nz = lambda X: X / np.linalg.norm(X, axis=1, keepdims=True).clip(1e-9)
S_raw_norm = nz(S_raw)  # [3769, 384] - MVP (văn bản gốc)
S_llm_norm = nz(S_llm)  # [3769, 384] - Ablation (chỉ hồ sơ LLM 5 chiều)
S_llm_3d_norm = nz(S_llm_3d)  # [3769, 384] - Ablation (LLM 3 chiều)
S_fused = np.concatenate([S_raw_norm, S_llm_norm], axis=1)  # [3769, 768] - MVP chính (Gốc + LLM)
S_fused_3d = np.concatenate([S_raw_norm, S_llm_3d_norm], axis=1)  # [3769, 768] - Ablation (Gốc + LLM 3 chiều)

assert S_raw_norm.shape == (3769, 384) and S_fused.shape == (3769, 768)

# Chuẩn bị mã nhóm trùng lặp dup_group_id và mặt nạ ứng viên bool
_, dup_int = np.unique(llm_meta["dup_group_id"], return_inverse=True)
dup_group_tensor = torch.tensor(dup_int, dtype=torch.long, device=DEVICE)

cold_mask_all = to_bool_mask(cold_val_ids + cold_test_ids, item2idx, 3769).to(DEVICE)
cand_val_a = to_bool_mask(cold_val_ids, item2idx, 3769)
cand_test_a = to_bool_mask(cold_test_ids, item2idx, 3769)
cand_test_b = torch.ones(3769, dtype=torch.bool)

# Tách 2 nhóm khóa cold-test cho P-RQ3 (124 khóa trùng / 378 khóa mới)
warm_dup_groups = set(dup_int[warm_items_np].tolist())
cold_test_dup_ids = [c for c in cold_test_ids if dup_int[item2idx[str(c)]] in warm_dup_groups]
cold_test_new_ids = [c for c in cold_test_ids if dup_int[item2idx[str(c)]] not in warm_dup_groups]
assert len(cold_test_dup_ids) == 124 and len(cold_test_new_ids) == 378, "Sai số lượng 124/378 khóa ứng viên!"

groups_prq3 = {
    "trung": {item2idx[str(c)] for c in cold_test_dup_ids},
    "moi": {item2idx[str(c)] for c in cold_test_new_ids},
}

te_courses_str = te_df["course_id"].astype(str)
dup_set_str = set(str(c) for c in cold_test_dup_ids)
new_set_str = set(str(c) for c in cold_test_new_ids)
print(
    f"P-RQ3 Cold-Test Split: Ứng viên: {len(cold_test_dup_ids)} khóa trùng |"
    f" {len(cold_test_new_ids)} khóa mới hoàn toàn\n  -> Khóa có tương tác"
    f" test: {te_courses_str[te_courses_str.isin(dup_set_str)].nunique()} trùng"
    f" ({int(te_courses_str.isin(dup_set_str).sum())} lượt) |"
    f" {te_courses_str[te_courses_str.isin(new_set_str)].nunique()} mới"
    f" ({int(te_courses_str.isin(new_set_str).sum())} lượt)"
)

train_u_tensor = torch.tensor(u_tr, dtype=torch.long, device=DEVICE)
train_i_tensor = torch.tensor(i_tr, dtype=torch.long, device=DEVICE)


# Xây dựng từ điển đích và tách riêng tên hist_tr, hist_trva (không ghi đè)
def build_hist(df):
  targets = {}
  for u_str, grp in df.groupby(df["user_id"].astype(str)):
    if u_str in user2idx:
      targets[user2idx[u_str]] = [item2idx[str(c)] for c in grp["course_id"]]
  return targets


val_targets = build_hist(va_cold)
test_targets = build_hist(te_cold)
hist_tr = build_hist(tr_cold)  # mặt nạ khi tinh chỉnh (cold-val)
hist_trva = build_hist(pd.concat([tr_cold, va_cold], ignore_index=True))  # mặt nạ khi đánh giá cold-test
mask_for_val = hist_tr
mask_for_test = hist_trva

assert len(val_targets) == 2092 and len(test_targets) == 3340

g_u_t = torch.tensor(g_u_np, device=DEVICE)
g_i_t = torch.tensor(g_i_np, device=DEVICE)


# Kiểm tra cơ bản và test overfit nhánh văn bản 
def run_sanity_and_overfit_tests():
  set_seed(42)
  sem_t = torch.tensor(S_fused, device=DEVICE)
  negs = sample_neg(100_000, warm_items_tensor).cpu().numpy()
  assert len(set(negs).intersection(set(cold_idx))) == 0, "LỖI: sample_neg lấy nhầm khóa lạnh!"

  g_i_dirty = g_i_t.clone()
  g_i_dirty[cold_idx] = 999.0
  test_model = MVP(g_u_t, g_i_dirty, sem_t, d=64, p_drop=0.3, freeze_backbone=True).to(DEVICE)
  test_model.eval()
  with torch.no_grad():
    h_eval, s_eval, _ = test_model.item_rep(cold_mask=cold_mask_all)
    assert torch.allclose(h_eval[cold_idx], s_eval[cold_idx], atol=1e-6), "LỖI: Khóa lạnh vẫn dính g_i!"

  set_seed(42)
  m = MVP(
      g_u_t,
      g_i_t,
      torch.tensor(S_fused, device=DEVICE),
      p_drop=1.0,
      freeze_backbone=True,
  ).to(DEVICE)
  opt = torch.optim.AdamW(m.proj.parameters(), lr=1e-2, weight_decay=0.0)
  b_u, b_pos = train_u_tensor[:512], train_i_tensor[:512]
  b_neg = sample_neg(512, warm_items_tensor)
  m.train()
  for _ in range(250):
    opt.zero_grad()
    h, _, _ = m.item_rep(cold_mask=None)
    loss = bpr_loss(m.g_u[b_u], h[b_pos], h[b_neg])
    loss.backward()
    opt.step()
  print("BPR (chỉ nhánh văn bản):", loss.item())
  assert (
      loss.item() < 0.05
  ), f"Test overfit nhánh văn bản thất bại! BPR loss = {loss.item():.4f}"
  print(
      "[PASS] Cả 3 kiểm tra (Sample Neg, Cold Mask, Overfit Nhánh Văn Bản với"
      f" loss={loss.item():.4f}) đều ĐẠT!"
  )


run_sanity_and_overfit_tests()

MD5 Checksums:
  g_u       : 7a170558642de73d301970e8ce7f0a86
  g_i       : d17ad9dfc4f4d6cb6de915c8649070d7
  raw_emb   : f80445649ef8a8217d048bd3e7ac0ad5
  llm_emb   : fc1819307be227a1544e1be0a4017393
  llm_3d_emb: 1af436095463ad0af3ef1203ec0ebd6c
  [Xác nhận] Bản g_i.npy trên đĩa đã có đúng 1368/3769 khóa có ||g_i|| = 0 (gồm 753 khóa lạnh + 615 khóa ấm không tương tác)!
P-RQ3 Cold-Test Split: Ứng viên: 124 khóa trùng | 378 khóa mới hoàn toàn
  -> Khóa có tương tác test: 56 trùng (1140 lượt) | 212 mới (4391 lượt)
BPR (chỉ nhánh văn bản): 1.1281000297458377e-05
[PASS] Cả 3 kiểm tra (Sample Neg, Cold Mask, Overfit Nhánh Văn Bản với loss=0.0000) đều ĐẠT!


In [6]:
os.makedirs("results_mvp/item_reps", exist_ok=True)
os.makedirs("scores", exist_ok=True)  # Thư mục lưu điểm từng học viên chuẩn hóa
SEEDS = [42, 2023, 2024, 1007, 999]

# Sinh 20 cấu hình ngẫu nhiên (seed 42)
space = dict(
    lr=[1e-4, 5e-4, 1e-3, 5e-3, 1e-2],
    p=[0.1, 0.3, 0.5, 0.8],
    lam=[0.01, 0.05, 0.1, 0.5],
    wd=[1e-5, 1e-4, 1e-3],
)
grid = [dict(zip(space, v)) for v in itertools.product(*space.values())]
trials = random.Random(42).sample(grid, 20)

# Danh sách Top-3 cấu hình có bộ siêu tham số chính (lr, p, lam) KHÁC NHAU trên cold-val (seed 42).
# Phía Gốc + LLM: Trial 03, 17, 18 có cùng (lr=1e-4, p=0.1, lam=0.1) chỉ khác wd, nên để kiểm tra
# độ bền thực sự giữa các cấu hình khác nhau, Top-3 phân biệt lấy Trial 03 (0.0923), Trial 14 (0.0917) và Trial 02 (0.0913).
CACHED_BEST_CFG_RAW = {"lr": 0.01, "p": 0.8, "lam": 0.01, "wd": 1e-05}
CACHED_TOP3_RAW_DISTINCT = [
    {"lr": 0.01, "p": 0.8, "lam": 0.01, "wd": 1e-05},  # Trial 12: Val 0.0953
    {"lr": 0.005, "p": 0.1, "lam": 0.1, "wd": 0.0001},  # Trial 15: Val 0.0953
    {"lr": 0.005, "p": 0.3, "lam": 0.1, "wd": 0.0001},  # Trial 01: Val 0.0951
]

CACHED_BEST_CFG_FUSED = {"lr": 0.0001, "p": 0.1, "lam": 0.1, "wd": 1e-05}
CACHED_TOP3_FUSED_DISTINCT = [
    {"lr": 0.0001, "p": 0.1, "lam": 0.1, "wd": 1e-05},  # Hạng 1 (Trial 03): Val seed 42 = 0.0923
    {"lr": 0.0001, "p": 0.3, "lam": 0.5, "wd": 0.0001},  # Hạng 2 phân biệt (lr,p,lam) (Trial 14): Val seed 42 = 0.0917
    {"lr": 0.0001, "p": 0.5, "lam": 0.05, "wd": 0.0001},  # Hạng 3 phân biệt (lr,p,lam) (Trial 02): Val seed 42 = 0.0913
]

CACHED_BEST_CFG_LLM = {"lr": 0.005, "p": 0.1, "lam": 0.1, "wd": 0.0001}
CACHED_TOP3_LLM_DISTINCT = [
    {"lr": 0.005, "p": 0.1, "lam": 0.1, "wd": 0.0001},   # Hạng 1 (Trial 15): Val seed 42 = 0.0998
    {"lr": 0.0001, "p": 0.1, "lam": 0.1, "wd": 1e-05},   # Hạng 2 (Trial 03): Val seed 42 = 0.0979
    {"lr": 0.0001, "p": 0.3, "lam": 0.5, "wd": 0.0001},  # Hạng 3 phân biệt (lr,p,lam) (Trial 14): Val seed 42 = 0.0974
]

def train_mvp(
    sem_np: np.ndarray,
    cfg: dict,
    seed: int = 42,
    freeze_backbone: bool = True,
    use_false_neg_mask: bool = True,
    batch_size: int = 4096,
    max_epochs: int = 300,
    eval_every: int = 5,
    patience: int = 10,
):
  set_seed(seed)
  sem_t = torch.tensor(sem_np, dtype=torch.float32, device=DEVICE)
  model = MVP(
      g_u_init=g_u_t,
      g_i_init=g_i_t,
      sem=sem_t,
      d=64,
      p_drop=cfg["p"],
      freeze_backbone=freeze_backbone,
  ).to(DEVICE)

  optimizer = torch.optim.AdamW(
      model.parameters(), lr=cfg["lr"], weight_decay=cfg["wd"]
  )
  n_edges = len(train_u_tensor)

  best_ndcg20 = -1.0
  best_epoch = 0
  best_state = copy.deepcopy(model.state_dict())
  no_improve = 0
  epoch_times = []

  for epoch in range(1, max_epochs + 1):
    t0 = time.perf_counter()
    model.train()
    perm = torch.randperm(n_edges, device=DEVICE)

    for start in range(0, n_edges, batch_size):
      idx = perm[start : start + batch_size]
      u_b = train_u_tensor[idx]
      pos_b = train_i_tensor[idx]
      neg_b = sample_neg(len(idx), warm_items_tensor)

      optimizer.zero_grad()
      h_item, s_proj, g_raw = model.item_rep(cold_mask=None)
      loss_bpr = bpr_loss(model.g_u[u_b], h_item[pos_b], h_item[neg_b])

      if cfg["lam"] > 0.0:
        unique_warm_b = torch.unique(torch.cat([pos_b, neg_b]))
        loss_al = align_loss(
            g=g_raw[unique_warm_b],
            s=s_proj[unique_warm_b],
            dup_group_id=dup_group_tensor[unique_warm_b],
            tau=0.2,
            use_false_neg_mask=use_false_neg_mask,
        )
        loss = loss_bpr + cfg["lam"] * loss_al
      else:
        loss = loss_bpr

      loss.backward()
      optimizer.step()

    epoch_times.append(time.perf_counter() - t0)

    if epoch % eval_every == 0:
      model.eval()
      with torch.no_grad():
        h_eval, _, _ = model.item_rep(cold_mask=cold_mask_all)
        u_eval = model.g_u
        score_fn = lambda ub: (u_eval[ub] @ h_eval.t()).cpu()
        val_res = evaluate(
            score_fn,
            val_targets,
            hist_tr,
            cand=cand_val_a,
            ks=(5, 10, 20),
            tie_seed=seed,
        )
        cur_ndcg20 = float(np.mean(val_res["NDCG@20"]))

      if cur_ndcg20 > best_ndcg20:
        best_ndcg20 = cur_ndcg20
        best_epoch = epoch
        best_state = copy.deepcopy(model.state_dict())
        no_improve = 0
      else:
        no_improve += 1
        if no_improve >= patience:
          break

  model.load_state_dict(best_state)
  return model, best_ndcg20, best_epoch, float(np.mean(epoch_times))


# Huấn luyện 5 seeds, lưu h_eval và in xác nhận chi tiết tập (b)
def tune_and_evaluate_variant(
    variant_name: str,
    tag: str,
    sem_matrix: np.ndarray,
    freeze_backbone: bool = True,
    run_20_trials: bool = False,
    fixed_best_cfg: dict = None,
):
  print(
      f"\n==================== {variant_name} (tag={tag},"
      f" input_dim={sem_matrix.shape[1]}) ===================="
  )
  trial_records = []
  if run_20_trials or fixed_best_cfg is None:
    best_cfg, best_val_score = None, -1.0
    for t_idx, cfg in enumerate(trials, 1):
      _, val_ndcg20, best_ep, sec_ep = train_mvp(
          sem_matrix, cfg, seed=42, freeze_backbone=freeze_backbone
      )
      trial_records.append({"trial": t_idx, "cfg": cfg, "val_ndcg20": val_ndcg20, "ep": best_ep})
      print(
          f"Trial {t_idx:02d}/20 | {cfg} -> Val NDCG@20: {val_ndcg20:.4f} (ep"
          f" {best_ep}, {sec_ep:.2f}s/ep)"
      )
      if val_ndcg20 > best_val_score:
        best_val_score = val_ndcg20
        best_cfg = cfg
    print(
        f"--> Cấu hình tốt nhất của {variant_name}: {best_cfg} (Val NDCG@20 ="
        f" {best_val_score:.4f})"
    )
  else:
    best_cfg = fixed_best_cfg
    print(
        "--> Sử dụng cấu hình đã chốt (tránh chọn lại sau khi xem test):"
        f" {best_cfg}"
    )

  results_a, results_b = [], []
  per_user_ndcg20_a = []
  n_trainable_params = 0

  for s in SEEDS:
    model, val_s, best_ep, sec_ep = train_mvp(
        sem_matrix, best_cfg, seed=s, freeze_backbone=freeze_backbone
    )
    n_trainable_params = sum(
        p.numel() for p in model.parameters() if p.requires_grad
    )
    model.eval()
    with torch.no_grad():
      h_eval, _, _ = model.item_rep(cold_mask=cold_mask_all)
      # Lưu biểu diễn khóa học h_eval theo seed [3769, 64]
      np.save(f"results_mvp/item_reps/{tag}_seed{s}.npy", h_eval.cpu().numpy())

      u_eval = model.g_u
      score_fn = lambda ub: (u_eval[ub] @ h_eval.t()).cpu()

      res_a = evaluate(
          score_fn,
          test_targets,
          hist_trva,
          cand=cand_test_a,
          ks=(5, 10, 20),
          tie_seed=s,
      )
      res_b = evaluate(
          score_fn,
          test_targets,
          hist_trva,
          cand=cand_test_b,
          ks=(5, 10, 20),
          tie_seed=s,
      )

      row_a = {k: float(np.mean(v)) for k, v in res_a.items()}
      row_a.update({
          "seed": s,
          "best_ep": best_ep,
          "val_ndcg20": val_s,
          "n_trainable_params": n_trainable_params,
          "sec_per_epoch": sec_ep,
      })
      results_a.append(row_a)

      row_b = {k: float(np.mean(v)) for k, v in res_b.items()}
      row_b.update({
          "seed": s,
          "best_ep": best_ep,
          "n_trainable_params": n_trainable_params,
          "sec_per_epoch": sec_ep,
          "n_users_hit20_b": int((np.asarray(res_b["HR@20"]) > 0).sum()),
      })
      results_b.append(row_b)

      per_user_ndcg20_a.append(res_a["NDCG@20"])

  df_a = pd.DataFrame(results_a)
  df_b = pd.DataFrame(results_b)
  print(
      f"\n[KẾT QUẢ 5 SEEDS - TẬP (a)] {variant_name} (Tham số huấn luyện:"
      f" {n_trainable_params:,}):"
  )
  for col in [
      "Recall@5",
      "Recall@10",
      "Recall@20",
      "NDCG@5",
      "NDCG@10",
      "NDCG@20",
      "HR@20",
  ]:
    if col in df_a:
      print(
          f"  {col:10s}: {df_a[col].mean():.4f} ± {df_a[col].std(ddof=1):.4f}"
      )

  print(
      f"[KẾT QUẢ 5 SEEDS - TẬP (b)"
      f" g_u={md5(GU_PATH)[:8]} / g_i={md5(GI_PATH)[:8]})] {variant_name}:"
  )
  for col in ["Recall@20", "NDCG@20", "HR@20"]:
    if col in df_b:
      print(
          f"  {col:10s}: {df_b[col].mean():.4f} ± {df_b[col].std(ddof=1):.4f}"
      )
  print(
      "  Chi tiết từng seed trên tập (b):"
      + ", ".join([
          f"seed {int(r['seed'])}: R@20={r['Recall@20']:.4f},"
          f" NDCG@20={r['NDCG@20']:.4f} (số HV trúng"
          f" top-20={int(r['n_users_hit20_b'])}/3340)"
          for _, r in df_b.iterrows()
      ])
  )

  mean_user_ndcg20 = np.mean(np.stack(per_user_ndcg20_a, axis=0), axis=0)
  assert mean_user_ndcg20.shape == (3340,)
  return best_cfg, df_a, df_b, mean_user_ndcg20, trial_records


# Đánh giá theo nhóm khóa cho P-RQ3
@torch.no_grad()
def eval_subgroups(
    score_fn, targets, mask, cand, groups, ks=(5, 10, 20), tie_seed=0, bs=1024
):
  """Xếp hạng trên TOÀN BỘ tập ứng viên cand (502 khóa cold-test).

  Với mỗi nhóm, chỉ số chỉ tính trên các đích thuộc nhóm, và chỉ cho học viên có
  ít nhất một đích trong nhóm.
  """
  users_sorted, K = sorted(targets), max(ks)
  g = torch.Generator().manual_seed(tie_seed)
  disc = 1.0 / np.log2(np.arange(2, K + 2))
  cand = torch.as_tensor(cand)
  out = {
      n: {f"{m}@{k}": [] for m in ("Recall", "NDCG", "HR") for k in ks}
      for n in groups
  }
  for s in range(0, len(users_sorted), bs):
    ub = users_sorted[s : s + bs]
    sc = score_fn(torch.tensor(ub)).float().cpu().clone()
    sc[:, ~cand] = -float("inf")
    for r, u in enumerate(ub):
      if mask.get(u):
        sc[r, list(mask[u])] = -float("inf")
    top = topk_random_ties(sc, K, g)  # cùng cách phá hòa với evaluate()
    for r, u in enumerate(ub):
      for n, gset in groups.items():
        gt = [i for i in targets[u] if i in gset]
        if not gt:
          continue
        hit = np.isin(top[r], gt).astype(float)
        for k in ks:
          h = hit[:k]
          idcg = disc[: min(len(gt), k)].sum()
          out[n][f"Recall@{k}"].append(h.sum() / len(gt))
          out[n][f"NDCG@{k}"].append((h * disc[:k]).sum() / idcg)
          out[n][f"HR@{k}"].append(float(h.sum() > 0))
  return {n: {m: np.array(v) for m, v in d.items()} for n, d in out.items()}


# Hàm chẩn đoán vì sao MVP bằng 0 trên tập (b)
@torch.no_grad()
def diagnose_cand_b(tag: str, seeds=SEEDS, bs=1024):
  users_sorted = sorted(test_targets)
  warm_all_mask = ~cold_mask_all.cpu()
  warm_active_mask = to_bool_mask([course_ids[i] for i in warm_items_np], item2idx, 3769)
  cold_test_mask = cand_test_a.cpu()
  rows = []

  for s in seeds:
    h_np = np.load(f"results_mvp/item_reps/{tag}_seed{s}.npy")
    h = torch.tensor(h_np, device=DEVICE)
    norm_h = torch.norm(h, dim=1).cpu()

    norm_warm_active = float(norm_h[warm_active_mask].mean())
    norm_warm_all = float(norm_h[warm_all_mask].mean())
    norm_cold_test = float(norm_h[cold_test_mask].mean())

    g = torch.Generator().manual_seed(s)
    warm_scores, cold_scores = [], []
    cold_in_top20 = 0
    total_top20 = len(users_sorted) * 20

    for start in range(0, len(users_sorted), bs):
      ub = users_sorted[start : start + bs]
      sc = (g_u_t[ub] @ h.t()).float().cpu()
      warm_scores.append(sc[:, warm_active_mask].mean().item())
      cold_scores.append(sc[:, cold_test_mask].mean().item())

      sc_masked = sc.clone()
      for r, u in enumerate(ub):
        if hist_trva.get(u):
          sc_masked[r, list(hist_trva[u])] = -float("inf")
      top20 = topk_random_ties(sc_masked, 20, g)
      cold_in_top20 += int(np.isin(top20, cold_idx).sum())

    rows.append({
        "seed": s,
        "norm_h_warm_active": norm_warm_active,
        "norm_h_warm_all": norm_warm_all,
        "norm_h_cold_test": norm_cold_test,
        "score_warm_mean": float(np.mean(warm_scores)),
        "score_cold_mean": float(np.mean(cold_scores)),
        "pct_cold_in_top20_b": 100.0 * cold_in_top20 / total_top20,
    })
  return pd.DataFrame(rows)


# Đường cong validation của MVP (đánh giá mỗi epoch từ 0 đến 30)
def train_one_epoch(model, opt, cfg, use_false_neg_mask=True, batch_size=4096):
  model.train()
  perm = torch.randperm(len(train_u_tensor), device=DEVICE)
  for start in range(0, len(perm), batch_size):
    idx = perm[start : start + batch_size]
    u_b, pos_b = train_u_tensor[idx], train_i_tensor[idx]
    neg_b = sample_neg(len(idx), warm_items_tensor)
    opt.zero_grad()
    h_item, s_proj, g_raw = model.item_rep(cold_mask=None)
    loss = bpr_loss(model.g_u[u_b], h_item[pos_b], h_item[neg_b])
    if cfg["lam"] > 0:
      w = torch.unique(torch.cat([pos_b, neg_b]))
      loss = loss + cfg["lam"] * align_loss(
          g_raw[w],
          s_proj[w],
          dup_group_tensor[w],
          tau=0.2,
          use_false_neg_mask=use_false_neg_mask,
      )
    loss.backward()
    opt.step()


def val_curve(sem_np, cfg, seed, n_epochs=30):
  set_seed(seed)
  model = MVP(
      g_u_t,
      g_i_t,
      torch.tensor(sem_np, device=DEVICE),
      d=64,
      p_drop=cfg["p"],
      freeze_backbone=True,
  ).to(DEVICE)
  opt = torch.optim.AdamW(
      model.parameters(), lr=cfg["lr"], weight_decay=cfg["wd"]
  )
  curve = []
  for epoch in range(0, n_epochs + 1):
    if epoch > 0:
      train_one_epoch(model, opt, cfg)
    model.eval()
    with torch.no_grad():
      h, _, _ = model.item_rep(cold_mask=cold_mask_all)
      r = evaluate(
          lambda ub: (model.g_u[ub] @ h.t()).cpu(),
          val_targets,
          hist_tr,
          cand=cand_val_a,
          ks=(20,),
          tie_seed=seed,
      )
    curve.append((epoch, float(np.mean(r["NDCG@20"]))))
  return curve


# Kiểm định Wilcoxon hai phía + Hiệu chỉnh Holm + Bootstrap CI 95% (kèm chênh lệch tương đối rel)
def compare(a, b, n_boot=10000, seed=0):
  assert a.shape == b.shape, f"Kích thước không khớp: {a.shape} vs {b.shape}"
  d = a - b
  rel = float(d.mean() / b.mean()) if abs(b.mean()) > 1e-12 else 0.0
  p = wilcoxon(a, b, zero_method="wilcox", alternative="two-sided").pvalue
  rng = np.random.default_rng(seed)
  boots = np.array(
      [d[rng.integers(0, len(d), len(d))].mean() for _ in range(n_boot)]
  )
  lo, hi = np.percentile(boots, [2.5, 97.5])
  return float(d.mean()), rel, float(lo), float(hi), int((d != 0).sum()), float(p)


def holm(pvals):
  pvals = np.asarray(pvals, dtype=np.float64)
  m = len(pvals)
  adj = np.empty(m, dtype=np.float64)
  run = 0.0
  for rank, i in enumerate(np.argsort(pvals)):
    run = max(run, (m - rank) * pvals[i])
    adj[i] = min(1.0, run)
  return adj


# Đo độ trễ suy luận trực tuyến trên GPU (khởi động kỹ + 20 reps để ổn định xung nhịp GPU)
@torch.no_grad()
def latency_ms_per_user(U, V, users, cand, K=20, bs=1024, reps=20):
  users_t = torch.tensor(users, device=DEVICE)
  cand_t = torch.as_tensor(cand, device=DEVICE)

  def run():
    for s in range(0, len(users_t), bs):
      sc = U[users_t[s : s + bs]] @ V.t()
      sc[:, ~cand_t] = -float("inf")
      sc.topk(K, dim=1)

  for _ in range(5):
    run()  # khởi động kỹ để GPU đạt xung nhịp ổn định
  times = []
  for _ in range(reps):
    if torch.cuda.is_available():
      torch.cuda.synchronize()
    t0 = time.perf_counter()
    run()
    if torch.cuda.is_available():
      torch.cuda.synchronize()
    times.append((time.perf_counter() - t0) * 1000 / len(users_t))
  return float(np.mean(times)), float(np.std(times, ddof=1))


# Đo độ trễ xen kẽ nhiều vòng theo thứ tự ngẫu nhiên, trả về trung vị
@torch.no_grad()
def interleaved_latency(models, users, cand, rounds=30, seed=0):
  # models: {tên: (U, V)}. Mỗi vòng đo mọi mô hình theo thứ tự ngẫu nhiên; trả về trung vị.
  rng = random.Random(seed)
  times = {k: [] for k in models}
  for U, V in models.values():
    latency_ms_per_user(U, V, users, cand, reps=3)  # khởi động
  for _ in range(rounds):
    order = list(models)
    rng.shuffle(order)
    for k in order:
      U, V = models[k]
      times[k].append(latency_ms_per_user(U, V, users, cand, reps=2)[0])
  return {k: float(np.median(v)) for k, v in times.items()}


print("Đã nạp xong toàn bộ các hàm huấn luyện và đánh giá!")

Đã nạp xong toàn bộ các hàm huấn luyện và đánh giá!


In [7]:
# Đặt RUN_20_TRIALS = False cho Raw và Fused để giữ nguyên cấu hình đã chốt trước khi xem test
RUN_20_TRIALS = False
RUN_MVP_RAW = True
RUN_MVP_FUSED = True
RUN_MVP_LLM = True  # Tinh chỉnh 20 cấu hình cho MVP chỉ hồ sơ LLM (cùng giao thức với DropoutNet, ALDI)
RUN_PERM_TEST = True
RUN_V3_DIAG = True

if RUN_MVP_RAW:
  best_cfg_raw, df_raw_a, df_raw_b, user_ndcg_raw, trials_raw = (
      tune_and_evaluate_variant(
          "MVP (Văn bản gốc)",
          "mvp_raw",
          S_raw_norm,
          freeze_backbone=True,
          run_20_trials=RUN_20_TRIALS,
          fixed_best_cfg=CACHED_BEST_CFG_RAW,
      )
  )
  df_raw_a.to_csv("results_mvp/mvp_raw_5seeds_cand_a.csv", index=False)
  df_raw_b.to_csv("results_mvp/mvp_raw_5seeds_cand_b.csv", index=False)
  np.save("results_mvp/user_ndcg20_mvp_raw.npy", user_ndcg_raw)
  np.save("scores/user_ndcg20_mvp_raw.npy", user_ndcg_raw) 
  json.dump(best_cfg_raw, open("results_mvp/best_cfg_raw.json", "w"), indent=2)

if RUN_MVP_FUSED:
  best_cfg_fused, df_fused_a, df_fused_b, user_ndcg_fused, trials_fused = (
      tune_and_evaluate_variant(
          "MVP (Văn bản gốc + Hồ sơ LLM)",
          "mvp_fused",
          S_fused,
          freeze_backbone=True,
          run_20_trials=RUN_20_TRIALS,
          fixed_best_cfg=CACHED_BEST_CFG_FUSED,
      )
  )
  df_fused_a.to_csv("results_mvp/mvp_fused_5seeds_cand_a.csv", index=False)
  df_fused_b.to_csv("results_mvp/mvp_fused_5seeds_cand_b.csv", index=False)
  np.save("results_mvp/user_ndcg20_mvp_fused.npy", user_ndcg_fused)
  np.save("scores/user_ndcg20_mvp_fused.npy", user_ndcg_fused)  
  json.dump(
      best_cfg_fused, open("results_mvp/best_cfg_fused.json", "w"), indent=2
  )

# MVP chỉ LLM (5 chiều): tinh chỉnh 20 cấu hình trên cold-val (seed 42), sau đó chạy 5 seeds
if RUN_MVP_LLM:
  cached_cfg_llm = (
      json.load(open("results_mvp/best_cfg_llm.json"))
      if os.path.exists("results_mvp/best_cfg_llm.json")
      else CACHED_BEST_CFG_LLM
  )
  best_cfg_llm, df_llm_a, df_llm_b, user_ndcg_llm, trials_llm = (
      tune_and_evaluate_variant(
          "MVP (Chỉ hồ sơ LLM)",
          "mvp_llm",
          S_llm_norm,
          freeze_backbone=True,
          run_20_trials=(cached_cfg_llm is None),
          fixed_best_cfg=cached_cfg_llm,
      )
  )
  df_llm_a.to_csv("results_mvp/mvp_llm_5seeds_cand_a.csv", index=False)
  df_llm_b.to_csv("results_mvp/mvp_llm_5seeds_cand_b.csv", index=False)
  np.save("results_mvp/user_ndcg20_mvp_llm.npy", user_ndcg_llm)
  np.save("scores/user_ndcg20_mvp_llm.npy", user_ndcg_llm)  
  json.dump(best_cfg_llm, open("results_mvp/best_cfg_llm.json", "w"), indent=2)
  if trials_llm:
    pd.DataFrame(trials_llm).to_csv("results_mvp/trials_mvp_llm.csv", index=False)

if RUN_PERM_TEST:
  cfg_main = json.load(open("results_mvp/best_cfg_fused.json"))
  perm = np.random.default_rng(42).permutation(3769)
  S_perm = S_fused[perm]
  model_perm, _, _, _ = train_mvp(S_perm, cfg_main, seed=42, freeze_backbone=True)
  model_perm.eval()
  with torch.no_grad():
    h_eval, _, _ = model_perm.item_rep(cold_mask=cold_mask_all)
    score_fn = lambda ub: (model_perm.g_u[ub] @ h_eval.t()).cpu()
    res_perm = evaluate(
        score_fn,
        test_targets,
        hist_trva,
        cand=cand_test_a,
        ks=(5, 10, 20),
        tie_seed=42,
    )
    perm_r20 = float(np.mean(res_perm["Recall@20"]))
    perm_ndcg20 = float(np.mean(res_perm["NDCG@20"]))
  print(
      f"\n[Kiểm tra hoán vị vector văn bản] Recall@20: {perm_r20:.4f} |"
      f" NDCG@20: {perm_ndcg20:.4f} (Tiêu chí ĐẠT: Recall@20 <= 0.0810)"
  )
  assert (
      perm_r20 <= 0.081
  ), f"KHÔNG ĐẠT kiểm tra hoán vị văn bản: Recall@20 = {perm_r20:.4f} > 0.081"

# Bảng chẩn đoán chuẩn ||h|| và điểm trung bình khóa ấm / khóa lạnh trên tập (b) (5 seeds)
if RUN_V3_DIAG:
  print("\n==================== CHẨN ĐOÁN TẬP (b) QUA 5 SEEDS ====================")
  for label, tag in [
      ("MVP – văn bản gốc", "mvp_raw"),
      ("MVP – chỉ hồ sơ LLM", "mvp_llm"),
      ("MVP – gốc + LLM", "mvp_fused"),
  ]:
    if not os.path.exists(f"results_mvp/item_reps/{tag}_seed42.npy"):
      continue
    df_d = diagnose_cand_b(tag, seeds=SEEDS)
    df_d.to_csv(f"results_mvp/v3_diag_{tag}_5seeds.csv", index=False)
    print(f"\n* {label}:")
    print(
        f"  ||h|| TB khóa ấm (2.401 khóa active) : {df_d['norm_h_warm_active'].mean():.2f} ± {df_d['norm_h_warm_active'].std(ddof=1):.2f}"
    )
    print(
        f"  ||h|| TB khóa ấm (3.016 khóa warm)   : {df_d['norm_h_warm_all'].mean():.2f} ± {df_d['norm_h_warm_all'].std(ddof=1):.2f}"
    )
    print(
        f"  ||h|| TB khóa cold-test (502 khóa)   : {df_d['norm_h_cold_test'].mean():.2f} ± {df_d['norm_h_cold_test'].std(ddof=1):.2f}"
    )
    print(
        f"  Điểm TB khóa ấm                      : {df_d['score_warm_mean'].mean():.3f} ± {df_d['score_warm_mean'].std(ddof=1):.3f}"
    )
    print(
        f"  Điểm TB khóa lạnh                    : {df_d['score_cold_mean'].mean():.3f} ± {df_d['score_cold_mean'].std(ddof=1):.3f}"
    )
    print(
        f"  Khóa lạnh trong top-20 (tập b)       : {df_d['pct_cold_in_top20_b'].mean():.2f}% ± {df_d['pct_cold_in_top20_b'].std(ddof=1):.2f}%"
    )
  print(
      "\nTập (b) đã được chạy lại "
      f" (MD5 g_u={md5(GU_PATH)[:8]},"
      f" g_i={md5(GI_PATH)[:8]}, trong đó 615 khóa ấm không tương tác đã có"
      " g_i=0). Sở dĩ kết quả tập (b) vẫn giữ nguyên (0.0062±0.0038 với MVP gốc"
      " và 0.0000 với MVP gốc+LLM) là vì 2.401 khóa ấm có tương tác có chuẩn"
      " ||h|| = 5.47 (so với 0.12 của khóa lạnh), nên top-20 luôn rơi vào"
      " nhóm 2.401 khóa ấm active này."
  )


==================== MVP (Văn bản gốc) (tag=mvp_raw, input_dim=384) ====================
--> Sử dụng cấu hình đã chốt (tránh chọn lại sau khi xem test): {'lr': 0.01, 'p': 0.8, 'lam': 0.01, 'wd': 1e-05}

[KẾT QUẢ 5 SEEDS - TẬP (a)] MVP (Văn bản gốc) (Tham số huấn luyện: 28,800):
  Recall@5  : 0.0651 ± 0.0050
  Recall@10 : 0.1026 ± 0.0137
  Recall@20 : 0.1681 ± 0.0219
  NDCG@5    : 0.0451 ± 0.0041
  NDCG@10   : 0.0576 ± 0.0070
  NDCG@20   : 0.0750 ± 0.0086
  HR@20     : 0.1985 ± 0.0248
[KẾT QUẢ 5 SEEDS - TẬP (b) g_u=7a170558 / g_i=d17ad9df)] MVP (Văn bản gốc):
  Recall@20 : 0.0062 ± 0.0038
  NDCG@20   : 0.0020 ± 0.0011
  HR@20     : 0.0068 ± 0.0041
  Chi tiết từng seed trên tập (b):seed 42: R@20=0.0023, NDCG@20=0.0008 (số HV trúng top-20=9/3340), seed 2023: R@20=0.0097, NDCG@20=0.0032 (số HV trúng top-20=35/3340), seed 2024: R@20=0.0018, NDCG@20=0.0008 (số HV trúng top-20=7/3340), seed 1007: R@20=0.0077, NDCG@20=0.0023 (số HV trúng top-20=29/3340), seed 999: R@20=0.0093, NDCG@20=0.0028 

In [8]:
print("\n==================== ĐÁNH GIÁ P-RQ3 (eval_subgroups) & ĐỒNG BỘ CONTENT-KNN ====================")

# Xây dựng hồ sơ học viên cho Content-KNN
def build_knn_user_profiles(
    S_item_np: np.ndarray,
    dedup_hist: bool = False,
    norm_user: bool = True,
) -> torch.Tensor:
  U_prof = np.zeros((len(users), S_item_np.shape[1]), dtype=np.float32)
  for u_idx, item_list in hist_tr.items():
    if item_list:
      idx_used = sorted(set(item_list)) if dedup_hist else item_list
      U_prof[u_idx] = S_item_np[idx_used].mean(axis=0)
  if norm_user:
    U_prof = nz(U_prof)
  return torch.tensor(U_prof, device=DEVICE)


# Chốt cố định một định nghĩa duy nhất cho Content-KNN (gốc + LLM):
# dùng đúng đầu vào s_i = [chuẩn hóa(gốc); chuẩn hóa(LLM)] (768 chiều), chuẩn hóa L2 ("concat_norm_each (MVP input)")
S_fused_knn = nz(S_fused)
print("Định nghĩa Content-KNN (gốc + LLM) đã chốt cố định (W5): 'concat_norm_each (MVP input)' (s_i = [norm(gốc); norm(LLM)], 768 chiều)")

knn_variants = {
    "knn_raw": (
        "Content-KNN – văn bản gốc",
        build_knn_user_profiles(S_raw_norm, False, True),
        torch.tensor(S_raw_norm, device=DEVICE),
    ),
    "knn_llm": (
        "Content-KNN – hồ sơ LLM",
        build_knn_user_profiles(S_llm_norm, False, True),
        torch.tensor(S_llm_norm, device=DEVICE),
    ),
    "knn_fused": (
        "Content-KNN – gốc + LLM",
        build_knn_user_profiles(S_fused_knn, False, True),
        torch.tensor(S_fused_knn, device=DEVICE),
    ),
}

knn_U_tensors, knn_V_tensors = {}, {}
for k_tag, (k_name, U_k, V_k) in knn_variants.items():
  knn_U_tensors[k_tag], knn_V_tensors[k_tag] = U_k, V_k
  fn_k = lambda ub, U=U_k, V=V_k: (U[ub.to(DEVICE)] @ V.t()).cpu()
  short_inp = k_tag.replace("knn_", "")  # raw, llm, fused

  # Đánh giá tập (a) để lưu điểm từng học viên cho Wilcoxon
  res_k_a = evaluate(
      fn_k, test_targets, hist_trva, cand=cand_test_a, ks=(5, 10, 20), tie_seed=0
  )
  np.save(f"results_mvp/user_ndcg20_{k_tag}.npy", res_k_a["NDCG@20"])
  np.save(f"results_mvp/user_r20_{k_tag}.npy", res_k_a["Recall@20"])
  np.save(f"scores/user_ndcg20_knn_{short_inp}.npy", res_k_a["NDCG@20"]) 

  # Đánh giá P-RQ3 theo nhóm khóa bằng eval_subgroups, lưu ĐỦ cả NDCG@20 và Recall@20
  res_k_sub = eval_subgroups(
      fn_k,
      test_targets,
      hist_trva,
      cand_test_a,
      groups_prq3,
      ks=(5, 10, 20),
      tie_seed=0,
  )
  np.save(f"results_mvp/user_ndcg20_{k_tag}_trung.npy", res_k_sub["trung"]["NDCG@20"])
  np.save(f"results_mvp/user_ndcg20_{k_tag}_moi.npy", res_k_sub["moi"]["NDCG@20"])
  np.save(f"results_mvp/user_r20_{k_tag}_trung.npy", res_k_sub["trung"]["Recall@20"])
  np.save(f"results_mvp/user_r20_{k_tag}_moi.npy", res_k_sub["moi"]["Recall@20"])
  # Lưu theo quy ước scores/user_ndcg20_{model}_{input}_{group}.npy
  np.save(f"scores/user_ndcg20_knn_{short_inp}_trung.npy", res_k_sub["trung"]["NDCG@20"])
  np.save(f"scores/user_ndcg20_knn_{short_inp}_moi.npy", res_k_sub["moi"]["NDCG@20"])

  if k_tag == "knn_raw":
    # Kiểm thử bắt buộc với Content-KNN văn bản gốc
    assert len(res_k_sub["trung"]["Recall@20"]) == 1028 and len(res_k_sub["moi"]["Recall@20"]) == 2844
    assert abs(res_k_sub["trung"]["Recall@20"].mean() - 0.2642) < 1e-3
    assert abs(res_k_sub["moi"]["Recall@20"].mean() - 0.1595) < 1e-3
    print("\n[PASS] Kiểm thử bắt buộc trên Content-KNN (văn bản gốc) ĐẠT!")

  print(
      f"  {k_name:28s} | (a) R@20={res_k_a['Recall@20'].mean():.4f},"
      f" NDCG@20={res_k_a['NDCG@20'].mean():.4f} | Nhóm TRÙNG (1028 HV):"
      f" R@20={res_k_sub['trung']['Recall@20'].mean():.4f},"
      f" NDCG@20={res_k_sub['trung']['NDCG@20'].mean():.4f} | Nhóm MỚI (2844"
      f" HV): R@20={res_k_sub['moi']['Recall@20'].mean():.4f},"
      f" NDCG@20={res_k_sub['moi']['NDCG@20'].mean():.4f}"
  )

# Tính P-RQ3 cho cả 3 biến thể MVP (văn bản gốc, chỉ hồ sơ LLM, gốc + LLM) từ biểu diễn khóa học h đã lưu
for m_name, tag in [
    ("MVP – văn bản gốc", "mvp_raw"),
    ("MVP – chỉ hồ sơ LLM", "mvp_llm"),
    ("MVP – gốc + LLM", "mvp_fused"),
]:
  if not os.path.exists(f"results_mvp/item_reps/{tag}_seed42.npy"):
    continue
  short_inp = tag.replace("mvp_", "")  # raw, llm, fused
  dup_list, new_list = [], []
  user_ndcg_trung_seeds, user_ndcg_moi_seeds = [], []
  user_r20_trung_seeds, user_r20_moi_seeds = [], []

  for s in SEEDS:
    h = torch.tensor(
        np.load(f"results_mvp/item_reps/{tag}_seed{s}.npy"), device=DEVICE
    )
    score_fn = lambda ub, h_t=h: (g_u_t[ub.to(DEVICE)] @ h_t.t()).cpu()
    res_sub = eval_subgroups(
        score_fn,
        test_targets,
        hist_trva,
        cand_test_a,
        groups_prq3,
        ks=(5, 10, 20),
        tie_seed=s,
    )

    dup_list.append(
        {k: float(np.mean(v)) for k, v in res_sub["trung"].items()}
    )
    new_list.append({k: float(np.mean(v)) for k, v in res_sub["moi"].items()})
    user_ndcg_trung_seeds.append(res_sub["trung"]["NDCG@20"])
    user_ndcg_moi_seeds.append(res_sub["moi"]["NDCG@20"])
    user_r20_trung_seeds.append(res_sub["trung"]["Recall@20"])
    user_r20_moi_seeds.append(res_sub["moi"]["Recall@20"])

  df_dup = pd.DataFrame(dup_list)
  df_new = pd.DataFrame(new_list)
  df_dup.to_csv(f"results_mvp/prq3_{tag}_trung_5seeds.csv", index=False)
  df_new.to_csv(f"results_mvp/prq3_{tag}_moi_5seeds.csv", index=False)

  mean_trung_ndcg = np.mean(np.stack(user_ndcg_trung_seeds, 0), 0)
  mean_moi_ndcg = np.mean(np.stack(user_ndcg_moi_seeds, 0), 0)

  np.save(f"results_mvp/user_ndcg20_{tag}_trung.npy", mean_trung_ndcg)
  np.save(f"results_mvp/user_ndcg20_{tag}_moi.npy", mean_moi_ndcg)
  np.save(
      f"results_mvp/user_r20_{tag}_trung.npy",
      np.mean(np.stack(user_r20_trung_seeds, 0), 0),
  )
  np.save(
      f"results_mvp/user_r20_{tag}_moi.npy",
      np.mean(np.stack(user_r20_moi_seeds, 0), 0),
  )
  # Lưu theo quy ước scores/user_ndcg20_mvp_{input}_{group}.npy
  np.save(f"scores/user_ndcg20_mvp_{short_inp}_trung.npy", mean_trung_ndcg)
  np.save(f"scores/user_ndcg20_mvp_{short_inp}_moi.npy", mean_moi_ndcg)

  print(
      f"  {m_name:28s} | Nhóm TRÙNG (1028 HV):"
      f" R@20={df_dup['Recall@20'].mean():.4f}±{df_dup['Recall@20'].std(ddof=1):.4f},"
      f" NDCG@20={df_dup['NDCG@20'].mean():.4f}±{df_dup['NDCG@20'].std(ddof=1):.4f}"
      " | Nhóm MỚI (2844 HV):"
      f" R@20={df_new['Recall@20'].mean():.4f}±{df_new['Recall@20'].std(ddof=1):.4f},"
      f" NDCG@20={df_new['NDCG@20'].mean():.4f}±{df_new['NDCG@20'].std(ddof=1):.4f}"
  )
# ==================== ĐỒNG BỘ SCORES VÀ TÍNH P-RQ3 CHO DROPOUTNET ====================
# Tự động tìm và copy mọi tệp user_ndcg20_*.npy từ /kaggle/input vào scores/ và results_mvp/
for root_dir in ["/kaggle/input", "results_dropoutnet", "results_aldi", "results_usim", "results_cgrc"]:
  if not os.path.exists(root_dir):
    continue
  for dirpath, _, filenames in os.walk(root_dir):
    for fn in filenames:
      if fn.startswith(("user_ndcg20_", "user_r20_")) and fn.endswith(".npy"):
        src_p = os.path.join(dirpath, fn)
        if fn.startswith("user_ndcg20_"):
          shutil.copy(src_p, os.path.join("scores", fn))
        shutil.copy(src_p, os.path.join("results_mvp", fn))

# Tìm đường dẫn u_e, v_e của DropoutNet (trong working dir hoặc /kaggle/input) và tính P-RQ3 
def find_rep_file(filename: str):
  for search_root in ["results_dropoutnet/item_reps", "/kaggle/input", "."]:
    if not os.path.exists(search_root):
      continue
    matches = glob.glob(os.path.join(search_root, "**", filename), recursive=True)
    if matches:
      return matches[0]
  return None

for name, label_dn in [
    ("raw", "DropoutNet – văn bản gốc"),
    ("llm", "DropoutNet – chỉ hồ sơ LLM"),
    ("fused", "DropoutNet – gốc + LLM"),
]:
  u_check = find_rep_file(f"u_e_{name}_seed42.npy")
  v_check = find_rep_file(f"v_e_{name}_seed42.npy")
  if u_check is None or v_check is None:
    print(f"  [Cảnh báo] Chưa tìm thấy u_e/v_e của {label_dn} cho P-RQ3. Hãy attach output của dropoutnet vào /kaggle/input!")
    continue

  per_seed_ndcg = {"trung": [], "moi": []}
  per_seed_r20 = {"trung": [], "moi": []}
  per_seed_a_ndcg = []
  dup_list_dn, new_list_dn = [], []

  for s in SEEDS:
    u_path = find_rep_file(f"u_e_{name}_seed{s}.npy")
    v_path = find_rep_file(f"v_e_{name}_seed{s}.npy")
    u_e = torch.tensor(np.load(u_path), device=DEVICE)
    v_e = torch.tensor(np.load(v_path), device=DEVICE)
    fn_dn = lambda ub, u_t=u_e, v_t=v_e: (u_t[ub.to(DEVICE)] @ v_t.t()).cpu()

    # Nếu chưa có tệp điểm tập (a) của DropoutNet trong scores/, tính và lưu luôn từ u_e, v_e
    if not os.path.exists(f"scores/user_ndcg20_dropoutnet_{name}.npy"):
      res_a_dn = evaluate(fn_dn, test_targets, hist_trva, cand=cand_test_a, ks=(5, 10, 20), tie_seed=s)
      per_seed_a_ndcg.append(res_a_dn["NDCG@20"])

    # Đánh giá phân nhóm P-RQ3 theo đúng Phụ lục A.3
    res_sub_dn = eval_subgroups(
        fn_dn, test_targets, hist_trva, cand_test_a, groups_prq3, ks=(5, 10, 20), tie_seed=s
    )
    dup_list_dn.append({k: float(np.mean(v)) for k, v in res_sub_dn["trung"].items()})
    new_list_dn.append({k: float(np.mean(v)) for k, v in res_sub_dn["moi"].items()})
    for g in ["trung", "moi"]:
      per_seed_ndcg[g].append(res_sub_dn[g]["NDCG@20"])
      per_seed_r20[g].append(res_sub_dn[g]["Recall@20"])

  if per_seed_a_ndcg:
    mean_a_dn = np.mean(np.stack(per_seed_a_ndcg, 0), 0)
    np.save(f"scores/user_ndcg20_dropoutnet_{name}.npy", mean_a_dn)
    np.save(f"results_mvp/user_ndcg20_dropoutnet_{name}.npy", mean_a_dn)

  for g in ["trung", "moi"]:
    mean_ndcg_g = np.mean(np.stack(per_seed_ndcg[g], 0), 0)
    mean_r20_g = np.mean(np.stack(per_seed_r20[g], 0), 0)
    np.save(f"scores/user_ndcg20_dropoutnet_{name}_{g}.npy", mean_ndcg_g)
    np.save(f"results_mvp/user_ndcg20_dropoutnet_{name}_{g}.npy", mean_ndcg_g)
    np.save(f"results_mvp/user_r20_dropoutnet_{name}_{g}.npy", mean_r20_g)

  df_dup_dn, df_new_dn = pd.DataFrame(dup_list_dn), pd.DataFrame(new_list_dn)
  print(
      f"  {label_dn:28s} | Nhóm TRÙNG (1028 HV):"
      f" R@20={df_dup_dn['Recall@20'].mean():.4f}±{df_dup_dn['Recall@20'].std(ddof=1):.4f},"
      f" NDCG@20={df_dup_dn['NDCG@20'].mean():.4f}±{df_dup_dn['NDCG@20'].std(ddof=1):.4f}"
      " | Nhóm MỚI (2844 HV):"
      f" R@20={df_new_dn['Recall@20'].mean():.4f}±{df_new_dn['Recall@20'].std(ddof=1):.4f},"
      f" NDCG@20={df_new_dn['NDCG@20'].mean():.4f}±{df_new_dn['NDCG@20'].std(ddof=1):.4f}"
  )
# ==========================================================================================================

# Đo độ trễ suy luận xen kẽ nhiều vòng (30 rounds, thứ tự ngẫu nhiên, lấy trung vị)
print("\n==================== ĐO ĐỘ TRỄ SUY LUẬN XEN KẼ P-RQ4 ====================")
users_eval_sorted = sorted(test_targets)
h_raw_s42 = torch.tensor(np.load("results_mvp/item_reps/mvp_raw_seed42.npy"), device=DEVICE)
h_fused_s42 = torch.tensor(np.load("results_mvp/item_reps/mvp_fused_seed42.npy"), device=DEVICE)

latency_models_dict = {
    "LightGCN (64d)": (g_u_t, g_i_t),
    "Content-KNN – văn bản gốc (384d)": (
        knn_U_tensors["knn_raw"],
        knn_V_tensors["knn_raw"],
    ),
    "Content-KNN – gốc + LLM (768d)": (
        knn_U_tensors["knn_fused"],
        knn_V_tensors["knn_fused"],
    ),
    "MVP – văn bản gốc (64d)": (g_u_t, h_raw_s42),
    "MVP – gốc + LLM (64d)": (g_u_t, h_fused_s42),
}
if os.path.exists("results_mvp/item_reps/mvp_llm_seed42.npy"):
  h_llm_s42 = torch.tensor(np.load("results_mvp/item_reps/mvp_llm_seed42.npy"), device=DEVICE)
  latency_models_dict["MVP – chỉ hồ sơ LLM (64d)"] = (g_u_t, h_llm_s42)

median_lats = interleaved_latency(
    latency_models_dict, users_eval_sorted, cand_test_a, rounds=30, seed=0
)
lgcn_median = median_lats["LightGCN (64d)"]

lat_rows = []
for m_label, (U_mat, _) in latency_models_dict.items():
  med_ms = median_lats[m_label]
  ratio = med_ms / max(lgcn_median, 1e-12)
  note = (
      "Độ trễ tương đương LightGCN (~1.0x) vì cùng phép tích vô hướng 64 chiều"
      if U_mat.shape[1] == 64
      else f"{ratio:.2f}x LightGCN"
  )
  lat_rows.append({
      "model": m_label,
      "dim": int(U_mat.shape[1]),
      "latency_ms_median_30rounds": med_ms,
      "ratio_vs_lightgcn_raw": ratio,
      "interpretation": note,
  })
  print(
      f"  {m_label:35s}: trung vị = {med_ms:.5f} ms/học viên (Tỷ lệ đo:"
      f" {ratio:.2f}x -> {note})"
  )

pd.DataFrame(lat_rows).to_csv("results_mvp/prq4_latency_gpu.csv", index=False)
print(
    f"  Thời gian huấn luyện MVP (Gốc + LLM): {df_fused_a['sec_per_epoch'].mean():.4f} ± {df_fused_a['sec_per_epoch'].std(ddof=1):.4f} s/epoch"
)


# Chi phí sinh hồ sơ LLM từ tệp benchmark
def parse_llm_benchmark_shards():
  patterns = ["/kaggle/input/datasets/dimlngkiu/throughput/throughput_benchmark_shard*.json"]
  found = []
  for pat in patterns:
    found.extend(glob.glob(pat, recursive=True))
  found = sorted(set(found))
  if not found:
    print(" Không tìm thấy file throughput_benchmark_shard*.json trong /kaggle/input.")
    return
  tot_sec = tot_n = 0
  for fp in found:
    d = json.load(open(fp))
    n_c = d["num_courses_tested"]
    sec = n_c * d["avg_sec_per_course"]
    tot_sec += sec
    tot_n += n_c
    print(f"    + Đọc {os.path.basename(fp)}: {sec:.1f}s ({n_c} khóa, avg={d['avg_sec_per_course']:.4f}s/khóa)")
  print(
      f" Chi phí sinh hồ sơ LLM ngoại tuyến ({tot_n} khóa): {tot_sec / 3600:.2f} GPU-giờ,"
      f" {tot_sec / tot_n:.2f} giây/khóa (hai shard chạy song song, chạy 1 lần duy nhất)"
  )


parse_llm_benchmark_shards()


==================== ĐÁNH GIÁ P-RQ3 (eval_subgroups) & ĐỒNG BỘ CONTENT-KNN ====================
Định nghĩa Content-KNN (gốc + LLM) đã chốt cố định (W5): 'concat_norm_each (MVP input)' (s_i = [norm(gốc); norm(LLM)], 768 chiều)

[PASS] Kiểm thử bắt buộc trên Content-KNN (văn bản gốc) ĐẠT!
  Content-KNN – văn bản gốc    | (a) R@20=0.2002, NDCG@20=0.1053 | Nhóm TRÙNG (1028 HV): R@20=0.2642, NDCG@20=0.1341 | Nhóm MỚI (2844 HV): R@20=0.1595, NDCG@20=0.0810
  Content-KNN – hồ sơ LLM      | (a) R@20=0.1808, NDCG@20=0.0819 | Nhóm TRÙNG (1028 HV): R@20=0.1905, NDCG@20=0.0841 | Nhóm MỚI (2844 HV): R@20=0.1526, NDCG@20=0.0691
  Content-KNN – gốc + LLM      | (a) R@20=0.1997, NDCG@20=0.1054 | Nhóm TRÙNG (1028 HV): R@20=0.2325, NDCG@20=0.1149 | Nhóm MỚI (2844 HV): R@20=0.1625, NDCG@20=0.0863
  MVP – văn bản gốc            | Nhóm TRÙNG (1028 HV): R@20=0.2088±0.0271, NDCG@20=0.0988±0.0146 | Nhóm MỚI (2844 HV): R@20=0.1373±0.0201, NDCG@20=0.0571±0.0059
  MVP – chỉ hồ sơ LLM          | Nhóm TRÙNG (1028

In [9]:
RUN_ABLATIONS = True
RUN_P_SENSITIVITY = True
RUN_V4_VAL_CURVE = False   # DỪNG CHẠY LẠI
RUN_V4_ROBUSTNESS = True   # BẬT ĐỂ LƯU TỆP _robust3s CHO RAW, FUSED VÀ CHẠY ĐỘ BỀN CHO LLM

cfg_raw = json.load(open("results_mvp/best_cfg_raw.json"))
cfg_main = json.load(open("results_mvp/best_cfg_fused.json"))
cfg_llm = (
    json.load(open("results_mvp/best_cfg_llm.json"))
    if os.path.exists("results_mvp/best_cfg_llm.json")
    else CACHED_BEST_CFG_LLM
)

if RUN_ABLATIONS:
  # Chạy các biến thể 3 chiều (E-RQ3) và 3 biến thể bỏ thành phần thực sự của MVP
  run_specs = [
      # Nhóm (a) - So sánh đầu vào (lưu tệp 3 chiều vào scores/ cho E-RQ3)
      ("ab1_llm_5d", "Chỉ dùng hồ sơ LLM (5 chiều)", S_llm_norm, cfg_llm, True, "input"),
      ("ab2_llm_3d", "Chỉ dùng hồ sơ LLM (3 chiều - bỏ tiên quyết và tiếp nối)", S_llm_3d_norm, cfg_llm, True, "input"),
      ("ab3_fused_3d", "Văn bản gốc + Hồ sơ LLM (3 chiều)", S_fused_3d, cfg_main, True, "input"),
      # Nhóm (b) - Bỏ thành phần thực sự (cùng cấu hình gốc + LLM cfg_main)
      ("ab4_no_drop", "1. Bỏ ID-dropout (p = 0)", S_fused, {**cfg_main, "p": 0.0}, True, "component"),
      ("ab5_no_align", "2. Bỏ căn chỉnh tương phản (lam = 0)", S_fused, {**cfg_main, "lam": 0.0}, True, "component"),
      ("ab6_no_fnmask", "3. Bỏ mặt nạ âm giả trong InfoNCE", S_fused, cfg_main, False, "component"),
  ]

  input_summary, comp_summary = [], []
  # Thêm kết quả đã có của Văn bản gốc và Gốc + LLM (5 chiều) vào Bảng (a)
  for base_tag, base_label in [("mvp_raw", "Văn bản gốc"), ("mvp_fused", "Văn bản gốc + Hồ sơ LLM (5 chiều)")]:
    if os.path.exists(f"results_mvp/{base_tag}_5seeds_cand_a.csv"):
      df_b = pd.read_csv(f"results_mvp/{base_tag}_5seeds_cand_a.csv")
      input_summary.append({
          "tag": base_tag,
          "variant": base_label,
          "R@20_mean": df_b["Recall@20"].mean(),
          "R@20_std": df_b["Recall@20"].std(ddof=1),
          "NDCG@20_mean": df_b["NDCG@20"].mean(),
          "NDCG@20_std": df_b["NDCG@20"].std(ddof=1),
      })

  for ab_tag, ab_name, sem_mat, ab_cfg, use_mask, group_type in run_specs:
    ab_rows, ab_user_ndcg = [], []
    for s in SEEDS:
      if ab_tag == "ab1_llm_5d" and os.path.exists(f"results_mvp/item_reps/mvp_llm_seed{s}.npy"):
        h_eval = torch.tensor(np.load(f"results_mvp/item_reps/mvp_llm_seed{s}.npy"), device=DEVICE)
        np.save(f"results_mvp/item_reps/{ab_tag}_seed{s}.npy", h_eval.cpu().numpy())
      elif os.path.exists(f"results_mvp/item_reps/{ab_tag}_seed{s}.npy"):
        h_eval = torch.tensor(np.load(f"results_mvp/item_reps/{ab_tag}_seed{s}.npy"), device=DEVICE)
      else:
        model, _, _, _ = train_mvp(
            sem_mat, ab_cfg, seed=s, freeze_backbone=True, use_false_neg_mask=use_mask
        )
        model.eval()
        with torch.no_grad():
          h_eval, _, _ = model.item_rep(cold_mask=cold_mask_all)
          np.save(f"results_mvp/item_reps/{ab_tag}_seed{s}.npy", h_eval.cpu().numpy())

      with torch.no_grad():
        score_fn = lambda ub, h_t=h_eval: (g_u_t[ub.to(DEVICE)] @ h_t.t()).cpu()
        r_a = evaluate(score_fn, test_targets, hist_trva, cand=cand_test_a, ks=(5, 10, 20), tie_seed=s)
        ab_rows.append({k: float(np.mean(v)) for k, v in r_a.items()})
        ab_user_ndcg.append(r_a["NDCG@20"])

    df_ab = pd.DataFrame(ab_rows)
    df_ab.to_csv(f"results_mvp/{ab_tag}_5seeds_cand_a.csv", index=False)
    mean_ab_ndcg = np.mean(np.stack(ab_user_ndcg, 0), 0)
    np.save(f"results_mvp/user_ndcg20_{ab_tag}.npy", mean_ab_ndcg)

    if ab_tag == "ab2_llm_3d":
      np.save("scores/user_ndcg20_mvp_llm_3d.npy", mean_ab_ndcg)
    elif ab_tag == "ab3_fused_3d":
      np.save("scores/user_ndcg20_mvp_fused_3d.npy", mean_ab_ndcg)

    row_info = {
        "tag": ab_tag,
        "variant": ab_name,
        "R@20_mean": df_ab["Recall@20"].mean(),
        "R@20_std": df_ab["Recall@20"].std(ddof=1),
        "NDCG@20_mean": df_ab["NDCG@20"].mean(),
        "NDCG@20_std": df_ab["NDCG@20"].std(ddof=1),
    }
    if group_type == "input":
      input_summary.append(row_info)
    else:
      comp_summary.append(row_info)

  print("\n==================== BẢNG (a): SO SÁNH ĐẦU VÀO CỦA MVP (5 SEEDS) ====================")
  for r in input_summary:
    print(f"  {r['variant']:58s} | R@20: {r['R@20_mean']:.4f}±{r['R@20_std']:.4f} | NDCG@20: {r['NDCG@20_mean']:.4f}±{r['NDCG@20_std']:.4f}")
  pd.DataFrame(input_summary).to_csv("results_mvp/mvp_input_comparison_summary.csv", index=False)

  print("\n==================== BẢNG (b): BỎ THÀNH PHẦN CỦA MVP (CÙNG CẤU HÌNH GỐC + LLM, 5 SEEDS) ====================")
  for r in comp_summary:
    print(f"  {r['variant']:58s} | R@20: {r['R@20_mean']:.4f}±{r['R@20_std']:.4f} | NDCG@20: {r['NDCG@20_mean']:.4f}±{r['NDCG@20_std']:.4f}")
  print("  -> Nhận xét (X4): Ở cấu hình đã chọn, các thành phần của MVP không đóng góp đáng kể; lợi ích đến từ đầu vào hồ sơ.")
  pd.DataFrame(comp_summary).to_csv("results_mvp/ablation_components_summary.csv", index=False)

# Phân tích độ nhạy theo p in {0.1, 0.3, 0.5, 0.8}, kèm NDCG@20 trên cold-val của từng p
if RUN_P_SENSITIVITY:
  print("\n==================== PHÂN TÍCH ĐỘ NHẠY THEO p IN {0.1, 0.3, 0.5, 0.8} (KÈM COLD-VAL NDCG@20) ====================")
  p_sens_rows = []
  cached_fused_val_map = {}
  if os.path.exists("results_mvp/mvp_fused_5seeds_cand_a.csv"):
    df_f_cached = pd.read_csv("results_mvp/mvp_fused_5seeds_cand_a.csv")
    cached_fused_val_map = dict(zip(df_f_cached["seed"].astype(int), df_f_cached["val_ndcg20"].astype(float)))

  for p_val in [0.1, 0.3, 0.5, 0.8]:
    cfg_p = {**cfg_main, "p": p_val}
    rows_a, rows_b, cold_norms, user_ndcg_p, val_ndcgs = [], [], [], [], []
    for s in SEEDS:
      if (
          abs(p_val - cfg_main["p"]) < 1e-9
          and os.path.exists(f"results_mvp/item_reps/mvp_fused_seed{s}.npy")
          and s in cached_fused_val_map
      ):
        h_eval = torch.tensor(
            np.load(f"results_mvp/item_reps/mvp_fused_seed{s}.npy"),
            device=DEVICE,
        )
        val_s = cached_fused_val_map[s]
      else:
        model, val_s, _, _ = train_mvp(S_fused, cfg_p, seed=s, freeze_backbone=True)
        model.eval()
        with torch.no_grad():
          h_eval, _, _ = model.item_rep(cold_mask=cold_mask_all)
          np.save(
              f"results_mvp/item_reps/sens_p{p_val}_seed{s}.npy",
              h_eval.cpu().numpy(),
          )
      val_ndcgs.append(float(val_s))

      with torch.no_grad():
        cold_norms.append(
            float(torch.norm(h_eval[cand_test_a.to(DEVICE)], dim=1).mean().cpu())
        )
        score_fn = lambda ub, h_t=h_eval: (g_u_t[ub.to(DEVICE)] @ h_t.t()).cpu()
        ra = evaluate(
            score_fn,
            test_targets,
            hist_trva,
            cand=cand_test_a,
            ks=(5, 10, 20),
            tie_seed=s,
        )
        rb = evaluate(
            score_fn,
            test_targets,
            hist_trva,
            cand=cand_test_b,
            ks=(5, 10, 20),
            tie_seed=s,
        )
        rows_a.append({k: float(np.mean(v)) for k, v in ra.items()})
        rows_b.append({k: float(np.mean(v)) for k, v in rb.items()})
        user_ndcg_p.append(ra["NDCG@20"])

    np.save(
        f"results_mvp/user_ndcg20_sens_p{p_val}.npy",
        np.mean(np.stack(user_ndcg_p, 0), 0),
    )
    dfa, dfb = pd.DataFrame(rows_a), pd.DataFrame(rows_b)
    p_sens_rows.append({
        "p": p_val,
        "Val_NDCG@20_mean": float(np.mean(val_ndcgs)),
        "Val_NDCG@20_std": float(np.std(val_ndcgs, ddof=1)),
        "R@20_a_mean": dfa["Recall@20"].mean(),
        "R@20_a_std": dfa["Recall@20"].std(ddof=1),
        "NDCG@20_a_mean": dfa["NDCG@20"].mean(),
        "NDCG@20_a_std": dfa["NDCG@20"].std(ddof=1),
        "R@20_b_mean": dfb["Recall@20"].mean(),
        "R@20_b_std": dfb["Recall@20"].std(ddof=1),
        "NDCG@20_b_mean": dfb["NDCG@20"].mean(),
        "NDCG@20_b_std": dfb["NDCG@20"].std(ddof=1),
        "norm_h_cold_mean": float(np.mean(cold_norms)),
    })
    print(
        f"  p = {p_val:.1f} | Cold-Val NDCG@20: {np.mean(val_ndcgs):.4f}±{np.std(val_ndcgs, ddof=1):.4f}"
        f" | Test (a) R@20: {dfa['Recall@20'].mean():.4f}±{dfa['Recall@20'].std(ddof=1):.4f},"
        f" NDCG@20: {dfa['NDCG@20'].mean():.4f}±{dfa['NDCG@20'].std(ddof=1):.4f} | (b)"
        f" R@20: {dfb['Recall@20'].mean():.4f}±{dfb['Recall@20'].std(ddof=1):.4f},"
        f" NDCG@20: {dfb['NDCG@20'].mean():.4f}±{dfb['NDCG@20'].std(ddof=1):.4f} |"
        f" ||h_cold||={np.mean(cold_norms):.2f}"
    )
  pd.DataFrame(p_sens_rows).to_csv(
      "results_mvp/v5_p_sensitivity.csv", index=False
  )
  print(
      " Bảng chính giữ nguyên cấu hình đã chốt trên seed 42 (p = 0.1, Cold-Val 5-seed = 0.0877, Test NDCG@20 = 0.0892)."
      " Lưu ý: nếu chọn theo trung bình Cold-Val 5 seeds thì p = 0.5 đạt cao nhất (0.0906±0.0015), không phải p = 0.8 (0.0888±0.0010)."
      " Không dùng kết quả của p = 0.8 (0.0974) hay p = 0.5 (0.0938) làm kết quả chính."
      " Khi p tăng từ 0.1 lên 0.8, chuẩn của f(s) trên khóa lạnh tăng từ 0.12 lên 4.60: ID-dropout buộc nhánh văn bản mang tín hiệu;"
      " ablation 'p = 0' không cho thấy điều này vì cấu hình chính có p = 0.1 rất nhỏ."
  )

# Đường cong validation mỗi epoch từ 0 đến 30 (5 seeds)
if RUN_V4_VAL_CURVE:
  print("\n==================== ĐƯỜNG CONG VALIDATION EPOCH 0..30 (5 SEEDS) ====================")
  cfg_raw = json.load(open("results_mvp/best_cfg_raw.json"))
  curve_rows = []
  for v_label, sem_mat, v_cfg in [
      ("MVP – văn bản gốc", S_raw_norm, cfg_raw),
      ("MVP – gốc + LLM", S_fused, cfg_main),
  ]:
    all_seed_curves = []
    for s in SEEDS:
      c = val_curve(sem_mat, v_cfg, seed=s, n_epochs=30)
      all_seed_curves.append([sc for _, sc in c])
    arr = np.array(all_seed_curves)  # [5, 31]
    for ep in range(31):
      curve_rows.append({
          "variant": v_label,
          "epoch": ep,
          "val_ndcg20_mean": float(arr[:, ep].mean()),
          "val_ndcg20_std": float(arr[:, ep].std(ddof=1)),
      })
    key_eps = [0, 1, 2, 3, 4, 5, 6, 8, 10, 15, 20, 25, 30]
    summary_str = " | ".join([
        f"ep{e}:{arr[:, e].mean():.4f}±{arr[:, e].std(ddof=1):.4f}"
        for e in key_eps
    ])
    print(f"  {v_label:20s}: {summary_str}")
  pd.DataFrame(curve_rows).to_csv(
      "results_mvp/v4_val_curves_5seeds.csv", index=False
  )

# Kiểm tra độ bền của P-RQ2 trên Top-3 cấu hình CÓ (lr, p, lam) KHÁC NHAU
if RUN_V4_ROBUSTNESS:
  print("\n==================== PHÂN TÍCH ĐỘ BỀN CỦA MVP (TOP-3 CẤU HÌNH x 3 VAL SEEDS CHO CẢ 3 ĐẦU VÀO) ====================")
  VAL_3SEEDS = [42, 2023, 2024]
  robust_chosen = {}
  robust_log = {}

  for v_tag, v_name, sem_mat, top3_list in [
      ("raw", "MVP – văn bản gốc", S_raw_norm, CACHED_TOP3_RAW_DISTINCT),
      ("llm", "MVP – chỉ hồ sơ LLM", S_llm_norm, CACHED_TOP3_LLM_DISTINCT),
      ("fused", "MVP – gốc + LLM", S_fused, CACHED_TOP3_FUSED_DISTINCT),
  ]:
    best_3seed_mean, chosen_cfg = -1.0, None
    cand_logs = []
    print(f"* Đánh giá 3 seeds ({VAL_3SEEDS}) trên cold-val cho top-3 cấu hình của {v_name}:")
    for idx_c, c_cand in enumerate(top3_list, 1):
      v_scores = [
          float(train_mvp(sem_mat, c_cand, seed=sv, freeze_backbone=True)[1])
          for sv in VAL_3SEEDS
      ]
      m_v, s_v = float(np.mean(v_scores)), float(np.std(v_scores, ddof=1))
      cand_logs.append({"cfg": c_cand, "val_3seed": v_scores, "mean": m_v, "std": s_v})
      print(f"  Top-{idx_c}: {c_cand} -> Val NDCG@20 (3 seeds) = {m_v:.4f} ± {s_v:.4f}")
      if m_v > best_3seed_mean:
        best_3seed_mean = m_v
        chosen_cfg = c_cand

    # Đánh giá cấu hình được chọn qua 5 seeds trên cold-test và lưu tệp _robust3s
    test_ndcgs, test_r20s, user_arrs = [], [], []
    for s in SEEDS:
      model, _, _, _ = train_mvp(sem_mat, chosen_cfg, seed=s, freeze_backbone=True)
      model.eval()
      with torch.no_grad():
        h_eval, _, _ = model.item_rep(cold_mask=cold_mask_all)
        r_a = evaluate(
            lambda ub: (model.g_u[ub] @ h_eval.t()).cpu(),
            test_targets,
            hist_trva,
            cand=cand_test_a,
            ks=(5, 10, 20),
            tie_seed=s,
        )
        test_ndcgs.append(float(np.mean(r_a["NDCG@20"])))
        test_r20s.append(float(np.mean(r_a["Recall@20"])))
        user_arrs.append(r_a["NDCG@20"])

    mean_user_rob = np.mean(np.stack(user_arrs, 0), 0)
    # Lưu tệp điểm chuẩn hóa _robust3s vào scores/ và results_mvp/ cho final_stats.ipynb (Mã X4, X1)
    np.save(f"scores/user_ndcg20_mvp_{v_tag}_robust3s.npy", mean_user_rob)
    np.save(f"results_mvp/user_ndcg20_mvp_{v_tag}_robust3s.npy", mean_user_rob)

    robust_chosen[v_tag] = {
        "chosen_cfg": chosen_cfg,
        "val_3seed_mean": best_3seed_mean,
        "test_r20_mean": float(np.mean(test_r20s)),
        "test_r20_std": float(np.std(test_r20s, ddof=1)),
        "test_ndcg20_mean": float(np.mean(test_ndcgs)),
        "test_ndcg20_std": float(np.std(test_ndcgs, ddof=1)),
        "user_ndcg20": mean_user_rob,
    }
    robust_log[v_tag] = {
        "candidates": cand_logs,
        "chosen_cfg": chosen_cfg,
        "test_ndcg20_mean": float(np.mean(test_ndcgs)),
        "test_ndcg20_std": float(np.std(test_ndcgs, ddof=1)),
    }
    print(
        f"  --> Cấu hình chọn theo 3-seed val của {v_name}: {chosen_cfg} | Test"
        f" R@20 = {np.mean(test_r20s):.4f}±{np.std(test_r20s, ddof=1):.4f} |"
        f" Test NDCG@20 = {np.mean(test_ndcgs):.4f}±{np.std(test_ndcgs, ddof=1):.4f}"
    )

  json.dump(robust_log, open("results_mvp/robust3s_mvp_log.json", "w"), indent=2)
  for inp_tag, label_cmp in [("fused", "Gốc + LLM"), ("llm", "Chỉ hồ sơ LLM")]:
    dm, rel, lo, hi, nz_cnt, p_rob = compare(
        robust_chosen[inp_tag]["user_ndcg20"], robust_chosen["raw"]["user_ndcg20"]
    )
    print(
        f"\n[Độ bền MVP: {label_cmp} vs. Văn bản gốc (3-seed val)] "
        f"diff = {dm:+.4f} ({rel*100:+.1f}%) [{lo:+.4f}, {hi:+.4f}], n≠0 = {nz_cnt}, p = {p_rob:.2e} "
        f"-> Giữ chiều (>0): {dm > 0}"
    )


==================== BẢNG (a): SO SÁNH ĐẦU VÀO CỦA MVP (5 SEEDS) ====================
  Văn bản gốc                                                | R@20: 0.1681±0.0219 | NDCG@20: 0.0750±0.0086
  Văn bản gốc + Hồ sơ LLM (5 chiều)                          | R@20: 0.2055±0.0131 | NDCG@20: 0.0892±0.0052
  Chỉ dùng hồ sơ LLM (5 chiều)                               | R@20: 0.2212±0.0091 | NDCG@20: 0.0978±0.0069
  Chỉ dùng hồ sơ LLM (3 chiều - bỏ tiên quyết và tiếp nối)   | R@20: 0.2002±0.0169 | NDCG@20: 0.0860±0.0082
  Văn bản gốc + Hồ sơ LLM (3 chiều)                          | R@20: 0.1860±0.0303 | NDCG@20: 0.0770±0.0129

==================== BẢNG (b): BỎ THÀNH PHẦN CỦA MVP (CÙNG CẤU HÌNH GỐC + LLM, 5 SEEDS) ====================
  1. Bỏ ID-dropout (p = 0)                                   | R@20: 0.2041±0.0122 | NDCG@20: 0.0887±0.0051
  2. Bỏ căn chỉnh tương phản (lam = 0)                       | R@20: 0.2013±0.0148 | NDCG@20: 0.0871±0.0100
  3. Bỏ mặt nạ âm giả trong InfoNCE            

In [10]:
print("\n==================== KIỂM ĐỊNH NỘI BỘ MVP (ABLATION & ĐỘ NHẠY p) ====================")

# Kiểm tra đồng thuận giữa Wilcoxon + Holm (kiểm định chính) và CI 95% Bootstrap
def check_agreement_note(lo: float, hi: float, pa: float) -> str:
  wilcox_sig = pa < 0.05
  ci_excludes_zero = (lo > 0.0) or (hi < 0.0)
  if wilcox_sig and ci_excludes_zero:
    return "Có ý nghĩa thống kê (p_Holm < 0.05)"
  elif (not wilcox_sig) and (not ci_excludes_zero):
    return "Không khác biệt có ý nghĩa"
  else:
    return "Không kết luận được (Wilcoxon và CI 95% không cùng kết luận)"


def run_hypothesis_family(
    family_name: str, ref_name: str, ref_arr: np.ndarray, comp_dict: dict
):
  print(f"\n--- {family_name} (Đối chiếu của: {ref_name}) ---")
  rows = {k: compare(ref_arr, v) for k, v in comp_dict.items()}
  p_adj = holm([r[-1] for r in rows.values()])
  out_records = []
  for (k, (dm, rel, lo, hi, nz_c, p)), pa in zip(rows.items(), p_adj):
    note_w9 = check_agreement_note(lo, hi, pa)
    print(
        f"  {ref_name} vs. {k:45s}: diff = {dm:+.4f} ({rel*100:+.1f}%) "
        f"(95% CI [{lo:+.4f}, {hi:+.4f}]), n≠0 = {nz_c:4d}, p = {p:.2e}, p_Holm = {pa:.2e} -> [{note_w9}]"
    )
    out_records.append({
        "family": family_name,
        "reference": ref_name,
        "target": k,
        "diff_mean": dm,
        "rel_pct": rel * 100.0,
        "ci95_lo": lo,
        "ci95_hi": hi,
        "n_nonzero": nz_c,
        "p_raw": p,
        "p_holm": pa,
        "conclusion_w9": note_w9,
    })
  return out_records


all_stat_rows = []
mvp_fused_u = np.load("results_mvp/user_ndcg20_mvp_fused.npy")
mvp_raw_u = np.load("results_mvp/user_ndcg20_mvp_raw.npy")
mvp_llm_u = (
    np.load("results_mvp/user_ndcg20_mvp_llm.npy")
    if os.path.exists("results_mvp/user_ndcg20_mvp_llm.npy")
    else np.load("results_mvp/user_ndcg20_ab1_llm_5d.npy")
)
knn_raw_u = np.load("results_mvp/user_ndcg20_knn_raw.npy")

# Bảng (a): Kiểm định so sánh các đầu vào trên MVP (tạm thời nội bộ MVP; E-RQ1/E-RQ3 chính thức lấy từ final_stats.ipynb)
fam_inputs_fused = {"MVP (Văn bản gốc)": mvp_raw_u}
if os.path.exists("results_mvp/user_ndcg20_ab3_fused_3d.npy"):
  fam_inputs_fused["MVP (Gốc + LLM 3 chiều)"] = np.load("results_mvp/user_ndcg20_ab3_fused_3d.npy")

all_stat_rows.extend(
    run_hypothesis_family(
        "Bảng (a1) - So sánh đầu vào MVP (Gốc + LLM 5 chiều) [NDCG@20]",
        "MVP (Gốc + LLM 5 chiều)",
        mvp_fused_u,
        fam_inputs_fused,
    )
)

if os.path.exists("results_mvp/user_ndcg20_ab2_llm_3d.npy"):
  ab2_3d = np.load("results_mvp/user_ndcg20_ab2_llm_3d.npy")
  all_stat_rows.extend(
      run_hypothesis_family(
          "Bảng (a2) - So sánh đầu vào MVP (Chỉ LLM 5 chiều) [NDCG@20]",
          "MVP (Chỉ LLM 5 chiều)",
          mvp_llm_u,
          {
              "MVP (Chỉ LLM 3 chiều - bỏ tiên quyết & tiếp nối)": ab2_3d,
              "MVP (Văn bản gốc)": mvp_raw_u,
          },
      )
  )

# Bảng (b): Kiểm định 3 biến thể BỎ THÀNH PHẦN thực sự của MVP (cùng cấu hình cfg_main)
if os.path.exists("results_mvp/user_ndcg20_ab4_no_drop.npy"):
  fam_ab_comp = {
      "1. Bỏ ID-dropout (p = 0)": np.load("results_mvp/user_ndcg20_ab4_no_drop.npy"),
      "2. Bỏ căn chỉnh tương phản (lam = 0)": np.load("results_mvp/user_ndcg20_ab5_no_align.npy"),
      "3. Bỏ mặt nạ âm giả trong InfoNCE": np.load("results_mvp/user_ndcg20_ab6_no_fnmask.npy"),
  }
  all_stat_rows.extend(
      run_hypothesis_family(
          "Bảng (b) - Ablation Bỏ thành phần của MVP (cùng cấu hình Gốc + LLM) [NDCG@20]",
          "MVP (Gốc + LLM)",
          mvp_fused_u,
          fam_ab_comp,
      )
  )

# Kiểm định bổ sung cho phân tích cơ chế độ nhạy p = 0.8 (chỉ phục vụ phân tích)
if os.path.exists("results_mvp/user_ndcg20_sens_p0.8.npy"):
  p08_u = np.load("results_mvp/user_ndcg20_sens_p0.8.npy")
  all_stat_rows.extend(
      run_hypothesis_family(
          "Phân tích cơ chế Độ nhạy p = 0.8 (không phải kết quả chính của MVP) [NDCG@20]",
          "MVP (Gốc + LLM, p = 0.8)",
          p08_u,
          {
              "MVP (Gốc + LLM, cấu hình chính p = 0.1)": mvp_fused_u,
              "MVP (Văn bản gốc, p = 0.8)": mvp_raw_u,
              "Content-KNN (Văn bản gốc)": knn_raw_u,
          },
      )
  )

# Lưu tệp thống kê nội bộ của MVP 
pd.DataFrame(all_stat_rows).to_csv(
    "results_mvp/mvp_internal_ablation_stats.csv", index=False
)
print(
    "\n[HOÀN TẤT] Đã lưu thống kê nội bộ MVP vào results_mvp/mvp_internal_ablation_stats.csv"
    " và xuất đầy đủ các tệp điểm học viên vào thư mục scores/ để chạy final_stats.ipynb!"
)


==================== KIỂM ĐỊNH NỘI BỘ MVP (ABLATION & ĐỘ NHẠY p) ====================

--- Bảng (a1) - So sánh đầu vào MVP (Gốc + LLM 5 chiều) [NDCG@20] (Đối chiếu của: MVP (Gốc + LLM 5 chiều)) ---
  MVP (Gốc + LLM 5 chiều) vs. MVP (Văn bản gốc)                            : diff = +0.0142 (+19.0%) (95% CI [+0.0105, +0.0181]), n≠0 = 1429, p = 5.14e-12, p_Holm = 5.14e-12 -> [Có ý nghĩa thống kê (p_Holm < 0.05)]
  MVP (Gốc + LLM 5 chiều) vs. MVP (Gốc + LLM 3 chiều)                      : diff = +0.0122 (+15.8%) (95% CI [+0.0097, +0.0148]), n≠0 = 1409, p = 1.65e-19, p_Holm = 3.30e-19 -> [Có ý nghĩa thống kê (p_Holm < 0.05)]

--- Bảng (a2) - So sánh đầu vào MVP (Chỉ LLM 5 chiều) [NDCG@20] (Đối chiếu của: MVP (Chỉ LLM 5 chiều)) ---
  MVP (Chỉ LLM 5 chiều) vs. MVP (Chỉ LLM 3 chiều - bỏ tiên quyết & tiếp nối): diff = +0.0117 (+13.6%) (95% CI [+0.0093, +0.0142]), n≠0 = 1417, p = 1.90e-27, p_Holm = 3.80e-27 -> [Có ý nghĩa thống kê (p_Holm < 0.05)]
  MVP (Chỉ LLM 5 chiều) vs. MVP (Văn bản gốc)  